In [2]:
# =============================================================================
# SECTION 0: IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# =============================================================================
# SECTION 2: LOAD REAL DATASET — UNSW IoT Traffic Traces
# =============================================================================
#
# Dataset: Sivanathan et al. (2018) — 20 days of real IoT traffic
# Source:  https://iotanalytics.unsw.edu.au/iottraces.html  (MIT-0 License)
# Format: CSV with header (10 columns) — read directly from .zip
#
# Used columns:
#   eth.src   → source device MAC address → class label
#   IP.proto  → network protocol (6=TCP, 17=UDP, 1=ICMP)
#   port.src  → unified source port (TCP or UDP)
#   port.dst  → unified destination port (TCP or UDP)

import os
import glob

# Path to the CSV directory (relative to this notebook)
CSV_DIR = os.path.join('..', '..', 'Legacy', 'iottraces_dataset', 'csv')

class_names = ['Smart-Static', 'Sensors', 'Audio', 'Video', 'Others']

# ── MAC → class mapping (source: IIsy/trace_processing/replacement_numeric) ───
# Devices not listed here default to class 4 (Others).
MAC_TO_CLASS = {
    # Class 0 — Smart-Static (hubs and smart plugs)
    'ec:1a:59:7a:02:c5': 0, 'ec:1a:59:83:28:11': 0, 'ec:1a:59:79:f4:89': 0,
    'd0:73:d5:01:83:08': 0, '00:24:e4:44:68:44': 0, '74:6a:89:00:2e:25': 0,
    '74:c6:3b:29:d7:1d': 0, '84:f3:eb:52:42:db': 0,
    # Class 1 — Sensors (motion sensors, alarms, weather stations)
    '70:88:6b:10:0f:c6': 1, '70:ee:50:03:b8:ac': 1,
    '00:24:e4:20:28:c6': 1, '18:b4:30:25:be:e4': 1,
    # Class 2 — Audio (smart speakers, audio streamers)
    'f4:f5:d8:d4:eb:12': 2, '44:65:0d:56:cc:d3': 2,
    '18:b7:9e:02:20:44': 2, '28:c2:dd:ff:a5:2d': 2,
    # Class 3 — Video (IP cameras, Smart TV, baby monitors)
    '00:16:6c:ab:6b:88': 3, '88:4a:ea:31:66:9d': 3, '70:ee:50:18:34:43': 3,
    '7c:70:bc:5d:5e:dc': 3, '30:8c:fb:2f:e4:b2': 3, 'b4:75:0e:ec:e5:a9': 3,
    'f4:f2:6d:93:51:f1': 3, 'e0:76:d0:3f:00:ae': 3, 'f4:f5:d8:8f:0a:3c': 3,
}


def load_all_days(csv_dir):
    """
    Loads all CSV files from the UNSW IoT Traffic Traces dataset.
    Reads directly from .zip — no prior extraction is required.

    For each daily file:
      1. Loads only the 4 necessary columns (usecols) to minimize memory.
      2. Normalizes MAC to lowercase and assigns class (4=Others if MAC unknown).
      3. Discards packets without an IP layer (ARP, etc.).
      4. Fills empty ports with 0 (ICMP packets without TCP/UDP).

    Returns:
        DataFrame with columns: ip_proto (uint8), src_port (uint16),
                                dst_port (uint16), label (uint8)
    """
    zip_files = sorted(glob.glob(os.path.join(csv_dir, '*.csv.zip')))
    if not zip_files:
        raise FileNotFoundError(
            f"No .csv.zip files found in: {os.path.abspath(csv_dir)}\n"
            f"Verify that CSV_DIR points to Legacy/iottraces_dataset/csv/"
        )

    print(f"Loading {len(zip_files)} days from the UNSW IoT Traffic Traces dataset...")
    print(f"  {'File':<22} {'IP Packets':>12}")
    print("  " + "─" * 37)

    frames = []
    for zf in zip_files:
        day = os.path.basename(zf).replace('.csv.zip', '')

        df_day = pd.read_csv(
            zf,
            usecols=['eth.src', 'IP.proto', 'port.src', 'port.dst'],
            dtype={'IP.proto': 'float32', 'port.src': 'float32', 'port.dst': 'float32'},
            low_memory=False
        )

        # Assign class from MAC (normalized to lowercase); unknowns → class 4
        df_day['label'] = (df_day['eth.src'].str.lower()
                                            .map(MAC_TO_CLASS)
                                            .fillna(4)
                                            .astype('uint8'))
        df_day = df_day.drop(columns=['eth.src'])

        # Remove packets without IP layer (ARP, etc.) and adjust types
        df_day = df_day.dropna(subset=['IP.proto'])
        df_day['ip_proto'] = df_day['IP.proto'].astype('uint8')
        df_day['src_port'] = df_day['port.src'].fillna(0).astype('uint16')
        df_day['dst_port'] = df_day['port.dst'].fillna(0).astype('uint16')
        df_day = df_day[['ip_proto', 'src_port', 'dst_port', 'label']]

        frames.append(df_day)
        print(f"  {day:<22} {len(df_day):>12,}")

    df = pd.concat(frames, ignore_index=True)
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"  {'─'*37}")
    print(f"  {'Total':<22} {len(df):>12,}")
    print(f"  Memory in RAM: {mem_mb:.0f} MB")
    return df


# === LOAD ALL 20 DAYS OF THE REAL DATASET ===
df = load_all_days(CSV_DIR)

print(f"\nClass distribution:")
print("─" * 55)
for cls, name in enumerate(class_names):
    cnt = (df['label'] == cls).sum()
    pct = cnt / len(df) * 100
    bar = '█' * max(1, int(pct / 2))
    print(f"  Class {cls} ({name:>12}): {cnt:>10,}  ({pct:5.1f}%)  {bar}")

Loading 20 days from the UNSW IoT Traffic Traces dataset...
  File                     IP Packets
  ─────────────────────────────────────
  16-09-23                    802,581
  16-09-24                    667,295
  16-09-25                    407,734
  16-09-26                    449,646
  16-09-27                    416,352
  16-09-28                  1,893,525
  16-09-29                    605,732
  16-09-30                    673,414
  16-10-01                    602,174
  16-10-02                    480,741
  16-10-03                    538,827
  16-10-04                  2,213,498
  16-10-05                  1,490,588
  16-10-06                    595,156
  16-10-07                  1,219,398
  16-10-08                    487,395
  16-10-09                    440,170
  16-10-10                    461,009
  16-10-11                  1,860,140
  16-10-12                  4,755,679
  ─────────────────────────────────────
  Total                    21,061,054
  Memory in RAM: 126 MB


In [4]:
# =============================================================================
# SECTION 4: PREPROCESSING
# =============================================================================

FEATURE_NAMES = ['ip_proto', 'src_port', 'dst_port']
LABEL_COL = 'label'

X = df[FEATURE_NAMES].values
y = df[LABEL_COL].values

print("Selected features:", FEATURE_NAMES)
print(f"X shape: {X.shape}  (samples × features)")
print(f"y shape: {y.shape}")

# Stratified 70% train / 30% test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set:         {X_test.shape[0]} samples")

print("\nClass distribution in training set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls} ({class_names[cls]}): {cnt}")

print("\nClass distribution in test set:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls} ({class_names[cls]}): {cnt}")

Selected features: ['ip_proto', 'src_port', 'dst_port']
X shape: (21061054, 3)  (samples × features)
y shape: (21061054,)

Training set: 14742737 samples
Test set:         6318317 samples

Class distribution in training set:
  Class 0 (Smart-Static): 801623
  Class 1 (Sensors): 197412
  Class 2 (Audio): 425716
  Class 3 (Video): 2302296
  Class 4 (Others): 11015690

Class distribution in test set:
  Class 0 (Smart-Static): 343553
  Class 1 (Sensors): 84605
  Class 2 (Audio): 182449
  Class 3 (Video): 986699
  Class 4 (Others): 4721011


In [5]:
# =============================================================================
# SECTION 5.2: TRAINING WITH max_depth=3 (L3 exercise)
# =============================================================================
#
# We use max_depth=3 as in the L3 exercise.
# Equivalent to the reference tree configuration in L3/tree-one.txt
# (trained on real IoT traffic data)

MAX_DEPTH = 3

clf_L3 = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    criterion='gini',
    random_state=SEED
)
clf_L3.fit(X_train, y_train)

y_pred_train = clf_L3.predict(X_train)
y_pred_test  = clf_L3.predict(X_test)

print(f"Tree trained with max_depth={MAX_DEPTH}")
print(f"  Training Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"  Test Accuracy:        {accuracy_score(y_test,  y_pred_test):.4f}")
print(f"  Number of leaves:           {clf_L3.get_n_leaves()}")
print(f"  Total number of nodes:   {clf_L3.tree_.node_count}")

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)
for feat, imp in zip(FEATURE_NAMES, clf_L3.feature_importances_):
    bar = '█' * int(imp * 40)
    print(f"  {feat:15s}: {imp:.4f}  {bar}")

print("\nInterpretation:")
print("  - Feature importance measures how much each feature reduces Gini impurity.")
print("  - A high value indicates that the feature is highly discriminative.")
print("  - This justifies why src_port and dst_port are good IoT classifiers.")

Tree trained with max_depth=3
  Training Accuracy: 0.8427
  Test Accuracy:        0.8427
  Number of leaves:           8
  Total number of nodes:   15

FEATURE IMPORTANCE
  ip_proto       : 0.0000  
  src_port       : 0.7789  ███████████████████████████████
  dst_port       : 0.2211  ████████

Interpretation:
  - Feature importance measures how much each feature reduces Gini impurity.
  - A high value indicates that the feature is highly discriminative.
  - This justifies why src_port and dst_port are good IoT classifiers.


In [6]:
# =============================================================================
# SECTION 7.1: get_lineage FUNCTION (adapted from IIsy/iisy_sw/framework/Machinelearning.py)
# =============================================================================
# Copyright (c) 2019 Zhaoqi Xiong, Noa Zilberman (NetFPGA)
# Adapted for the 3-feature exercise: ip_proto, src_port, dst_port

def get_lineage(clf, feature_names):
    """
    Extracts the decision tree rules in readable format.

    Traverses the tree from each leaf up to the root, building
    the sequence of conditions that lead to that leaf.

    Args:
        clf:          Trained sklearn decision tree.
        feature_names: List of feature names ['ip_proto', 'src_port', 'dst_port'].

    Returns:
        rules:   List of strings with 'when ... then class' rules.
        thresholds_per_feature: Dict feature → list of thresholds (int).
    """
    tree  = clf.tree_
    left  = tree.children_left     # Left child index of each node (-1 if leaf)
    right = tree.children_right    # Right child index of each node
    thr   = tree.threshold          # Split threshold at each node
    feats = [feature_names[i] for i in tree.feature]  # Feature of each node
    value = tree.value              # Class distribution [n_nodes, 1, n_classes]

    # IDs of leaf nodes (children_left == -1 in sklearn)
    leaf_ids = np.argwhere(left == -1)[:, 0]

    def recurse(node, lineage=None):
        """Walks up from 'node' to the root, collecting the path."""
        if lineage is None:
            lineage = [node]  # Start with the current leaf
        if node in left:
            parent = int(np.where(left == node)[0][0])   # [0]=array from tuple, [0]=scalar index
            side = 'l'  # Came via the left branch (<=)
        else:
            parent = int(np.where(right == node)[0][0])  # same fix
            side = 'r'  # Came via the right branch (>)
        lineage.append((parent, side, thr[parent], feats[parent]))
        if parent == 0:
            lineage.reverse()  # From root to leaf
            return lineage
        return recurse(parent, lineage)

    rules = []
    thresholds_per_feature = {f: [] for f in feature_names}

    for leaf in leaf_ids:
        path  = recurse(leaf)
        clause = ' when '

        for node_info in path:
            # Leaves are simple integers (no tuple), internal nodes are 4-tuples
            if not isinstance(node_info, tuple):
                continue
            parent, side, threshold, feature = node_info

            sign = '<=' if side == 'l' else '>'
            # Show threshold as integer (sklearn uses 11.5, 2949.5, etc.)
            clause += f"{feature}{sign}{int(threshold)} and "

            # Collect thresholds per feature
            if threshold > -2:  # -2 is the value for leaf nodes in sklearn
                thresholds_per_feature[feature].append(int(threshold))

        # Predicted class of the leaf: index of the maximum value
        predicted_class = int(np.argmax(value[leaf][0]))

        # Remove the final ' and ' and add the class
        clause = clause.rstrip(' and ') + f' then {predicted_class}'
        rules.append(clause)

    # Deduplicate and sort thresholds
    for f in feature_names:
        thresholds_per_feature[f] = sorted(set(thresholds_per_feature[f]))

    return rules, thresholds_per_feature


# Extract rules from the L3 tree
rules_L3, thresholds_L3 = get_lineage(clf_L3, FEATURE_NAMES)

print(f"Tree L3 (max_depth=3): {len(rules_L3)} extracted rules")
print("\nThresholds per feature:")
for feat in FEATURE_NAMES:
    print(f"  {feat} = {thresholds_L3[feat]}")

Tree L3 (max_depth=3): 8 extracted rules

Thresholds per feature:
  ip_proto = []
  src_port = [547, 1899, 3071, 49280, 60633]
  dst_port = [67, 1917]


In [7]:
# =============================================================================
# SECTION 7.2: GENERATE TREE FILE (L3/tree-one.txt format)
# =============================================================================

def generate_tree_file_content(rules, thresholds_per_feature, feature_names):
    """
    Generates the tree file content in IIsy / DT exercise format.

    Format:
        ip_proto = [11];                        # thresholds for each feature
        src_port = [2949, 5019, ...];
        dst_port = [67];
         when src_port<=2949 and dst_port<=67  then 1;
         ...
    """
    lines = []
    for feat in feature_names:
        thrs = thresholds_per_feature[feat]
        lines.append(f"{feat} = {thrs};")
    lines.append('')
    for rule in rules:
        lines.append(rule + ";")
    return '\n'.join(lines)


tree_content_L3 = generate_tree_file_content(rules_L3, thresholds_L3, FEATURE_NAMES)

print("\n" + "=" * 60)
print("GENERATED TREE (tree.txt format — max_depth=3)")
print("=" * 60)
print(tree_content_L3)

# Save to file
with open('tree_L3_generado.txt', 'w') as f:
    f.write(tree_content_L3)

print(f"\nFile saved: tree_L3_generado.txt")


GENERATED TREE (tree.txt format — max_depth=3)
ip_proto = [];
src_port = [547, 1899, 3071, 49280, 60633];
dst_port = [67, 1917];

 when src_port<=3071 and dst_port<=67 and src_port<=547 then 1;
 when src_port<=3071 and dst_port<=67 and src_port>547 then 4;
 when src_port<=3071 and dst_port>67 and src_port<=1899 then 4;
 when src_port<=3071 and dst_port>67 and src_port>1899 then 4;
 when src_port>3071 and src_port<=49280 and dst_port<=1917 then 3;
 when src_port>3071 and src_port<=49280 and dst_port>1917 then 0;
 when src_port>3071 and src_port>49280 and src_port<=60633 then 4;
 when src_port>3071 and src_port>49280 and src_port>60633 then 3;

File saved: tree_L3_generado.txt


---
## Sección 8: Traducción del Árbol a Reglas de Rango para P4

### El problema de la traducción

Las tablas P4 con matching `range` aceptan entradas de la forma:
```
table_add feature2_exact set_actionselect2  <min>-><max>  =>  <índice>  1
```

El árbol de decisión genera condiciones del tipo `src_port <= 2949` o `src_port > 5019`. Necesitamos convertir estos **predicados del árbol** en **intervalos de rango** con un índice entero asignado.

### Algoritmo de traducción

**Paso 1**: Extraer todos los umbrales únicos de cada feature del árbol (ya hecho en Sección 7).

**Paso 2**: Crear intervalos a partir de los umbrales:

Dados umbrales $[t_1, t_2, \ldots, t_n]$ para un feature con valores en $[0, V_{max}]$:

| Índice | Intervalo | Condición equivalente del árbol |
|---|---|---|
| 1 | $[0, t_1]$ | $\text{feature} \leq t_1$ |
| 2 | $[t_1+1, t_2]$ | $\text{feature} > t_1 \wedge \text{feature} \leq t_2$ |
| 3 | $[t_2+1, t_3]$ | $\text{feature} > t_2 \wedge \text{feature} \leq t_3$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| $n+1$ | $[t_n+1, V_{max}]$ | $\text{feature} > t_n$ |

**Paso 3**: Para cada hoja del árbol, determinar qué índices de intervalo son compatibles con las condiciones del camino raíz→hoja.

**Paso 4**: Generar la entrada de la tabla de decisión `ipv4_exact` con rangos de índices de action_select.

In [8]:
# =============================================================================
# SECTION 8.1: CREATING RANGE INTERVALS
# =============================================================================

# Maximum values for each feature
FEATURE_MAX = {
    'ip_proto': 255,    # IP Protocol: 0-255
    'src_port': 65535,  # Source Port: 0-65535
    'dst_port': 65535   # Destination Port: 0-65535
}
FEATURE_NAMES = ['ip_proto', 'src_port', 'dst_port']
thresholds_L3 = {
    'ip_proto': [],
    'src_port': [547, 1899, 3071, 49280, 60633],
    'dst_port': [67, 1917]
}
def thresholds_to_intervals(thresholds, max_val):
    """
    Converts a list of thresholds into [min, max] intervals with indices.

    Example: thresholds=[11, 2949, 5019], max_val=65535
      → [(0, 11, 1), (12, 2949, 2), (2950, 5019, 3), (5020, 65535, 4)]

    Args:
        thresholds: List of sorted integer thresholds.
        max_val:    Maximum value of the feature.

    Returns:
        List of tuples (min_val, max_val, index).
    """
    if not thresholds:
        # No thresholds → a single interval for the entire range
        return [(0, max_val, 1)]

    intervals = []
    prev = 0
    for idx, t in enumerate(sorted(thresholds)):
        intervals.append((prev, t, idx + 1))
        prev = t + 1
    intervals.append((prev, max_val, len(thresholds) + 1))
    return intervals


# Create intervals for each feature of the L3 tree
intervals_L3 = {}
for feat in FEATURE_NAMES:
    intervals_L3[feat] = thresholds_to_intervals(thresholds_L3[feat], FEATURE_MAX[feat])

print("Range intervals per feature (L3 tree (real UNSW)):")
print("=" * 70)
feat_table_names = {
    'ip_proto': 'ip_proto - feature1_exact (Table 1)',
    'src_port': 'src_port - feature2_exact (Table 2)',
    'dst_port': 'dst_port - feature3_exact (Table 3)'
}
for feat in FEATURE_NAMES:
    print(f"\n► {feat_table_names[feat]}")
    print(f"  {'Index':>6}  {'Minimum':>8}  {'Maximum':>8}  {'Hex Min':>10}  {'Hex Max':>10}")
    print(f"  {'------':>6}  {'-------':>8}  {'-------':>8}  {'--------':>10}  {'--------':>10}")
    for lo, hi, idx in intervals_L3[feat]:
        print(f"  {idx:>6}  {lo:>8}  {hi:>8}  {hex(lo):>10}  {hex(hi):>10}")

Range intervals per feature (L3 tree (real UNSW)):

► ip_proto - feature1_exact (Table 1)
   Index   Minimum   Maximum     Hex Min     Hex Max
  ------   -------   -------    --------    --------
       1         0       255         0x0        0xff

► src_port - feature2_exact (Table 2)
   Index   Minimum   Maximum     Hex Min     Hex Max
  ------   -------   -------    --------    --------
       1         0       547         0x0       0x223
       2       548      1899       0x224       0x76b
       3      1900      3071       0x76c       0xbff
       4      3072     49280       0xc00      0xc080
       5     49281     60633      0xc081      0xecd9
       6     60634     65535      0xecda      0xffff

► dst_port - feature3_exact (Table 3)
   Index   Minimum   Maximum     Hex Min     Hex Max
  ------   -------   -------    --------    --------
       1         0        67         0x0        0x43
       2        68      1917        0x44       0x77d
       3      1918     65535       0x

---
## Sección 9: Generación de Comandos `table_add`

### Estructura del pipeline P4

```
Paquete llegando
      ↓
[feature1_exact]  ip_proto ∈ [lo, hi] → meta.action_select1 = índice₁
      ↓
[feature2_exact]  src_port ∈ [lo, hi] → meta.action_select2 = índice₂
      ↓
[feature3_exact]  dst_port ∈ [lo, hi] → meta.action_select3 = índice₃
      ↓
[ipv4_exact]  (índice₁, índice₂, índice₃) → clase → acción de forwarding
```

### Mapeo clase → acción

| Clase | Dispositivo | Acción |
|---|---|---|
| 0 | Smart-Static | Forward al puerto 2 (host H2) |
| 1 | Sensores | Forward al puerto 3 (host H3) |
| 2 | Audio | Forward al puerto 2 (host H2) |
| 3 | Video | Forward al puerto 3 (host H3) |
| 4 | Otros | Forward al puerto 4 (host H4) |

In [10]:
# =============================================================================
# SECTION 9.1: table_add COMMAND GENERATION ALGORITHM
# =============================================================================

import numpy as np
# Class → (next-hop IP, output port) mapping
# Based on the DT exercise topology (4 hosts: H1=control, H2, H3, H4)
CLASS_TO_ACTION = {
    0: {'nhop': '0x0a000102', 'port': 2},  # Smart-Static → H2 (10.0.1.2)
    1: {'nhop': '0x0a000103', 'port': 3},  # Sensors     → H3 (10.0.1.3)
    2: {'nhop': '0x0a000102', 'port': 2},  # Audio        → H2 (10.0.1.2)
    3: {'nhop': '0x0a000103', 'port': 3},  # Video        → H3 (10.0.1.3)
    4: {'nhop': '0x0a000104', 'port': 4},  # Others        → H4 (10.0.1.4)
}
class_names = ['Smart-Static', 'Sensors', 'Audio', 'Video', 'Others']
FEAT_TABLE  = {'ip_proto': 'feature1_exact', 'src_port': 'feature2_exact', 'dst_port': 'feature3_exact'}
FEAT_ACTION = {'ip_proto': 'set_actionselect1', 'src_port': 'set_actionselect2', 'dst_port': 'set_actionselect3'}


def get_compatible_indices(constraints, intervals):
    """
    Determines which interval indices are compatible with a set of constraints.

    A constraint is ('<=', t) or ('>', t) where t is an integer threshold.
    An interval [lo, hi] with index idx is compatible if all values
    in that interval satisfy ALL constraints.

    For '<=t': the interval is compatible if hi <= t (the entire interval is to the left).
    For '>t':  the interval is compatible if lo > t  (the entire interval is to the right).

    Args:
        constraints: List of ('<=', thr) or ('>', thr).
        intervals:   List of (lo, hi, idx).

    Returns:
        List of compatible indices.
    """
    # Calculate effective range of compatible values
    eff_lo = 0
    eff_hi = float('inf')

    for op, thr in constraints:
        t_int = int(thr)
        if op == '<=':
            eff_hi = min(eff_hi, t_int)
        else:  # '>'
            eff_lo = max(eff_lo, t_int + 1)

    if eff_hi == float('inf'):
        eff_hi = max(hi for _, hi, _ in intervals)

    # An interval is compatible if it is COMPLETELY within the effective range
    return [idx for lo, hi, idx in intervals if lo >= eff_lo and hi <= eff_hi]


def generate_all_table_add_commands(clf, feature_names, intervals_per_feature,
                                    class_to_action, thrift_port=9090):
    """
    Generates ALL necessary table_add commands for the P4 switch.

    Includes:
    - Commands for feature1_exact, feature2_exact, feature3_exact
    - Commands for ipv4_exact (final decision table)

    Returns:
        List of strings with shell commands.
    """
    tree    = clf.tree_
    left    = tree.children_left
    right   = tree.children_right
    thr     = tree.threshold
    feats   = [feature_names[i] for i in tree.feature]
    value   = tree.value

    CLI = f"simple_switch_CLI --thrift-port {thrift_port}"
    commands = []

    # ─── Feature tables: one entry per interval ───────────────────────
    commands.append(f"# ─── Table 1: ip_proto (Feature 1) ───")
    for lo, hi, idx in intervals_per_feature['ip_proto']:
        cmd = (f'{CLI} <<< "table_add MyIngress.feature1_exact '
               f'MyIngress.set_actionselect1 {lo}->{hi} => {idx} 1"')
        commands.append(cmd)

    commands.append("")
    commands.append(f"# ─── Table 2: src_port (Feature 2) ───")
    for lo, hi, idx in intervals_per_feature['src_port']:
        cmd = (f'{CLI} <<< "table_add MyIngress.feature2_exact '
               f'MyIngress.set_actionselect2 {lo}->{hi} => {idx} 1"')
        commands.append(cmd)

    commands.append("")
    commands.append(f"# ─── Table 3: dst_port (Feature 3) ───")
    for lo, hi, idx in intervals_per_feature['dst_port']:
        cmd = (f'{CLI} <<< "table_add MyIngress.feature3_exact '
               f'MyIngress.set_actionselect3 {lo}->{hi} => {idx} 1"')
        commands.append(cmd)

    # ─── ipv4_exact Decision Table ────────────────────────────────────────
    commands.append("")
    commands.append("# ─── Table 4: ipv4_exact (Final Decision) ───")

    leaf_ids = np.argwhere(left == -1)[:, 0]
    seen = set()  # To avoid duplicate entries

    for leaf in leaf_ids:
        # Reconstruct path from root to leaf
        path_constraints = {feat: [] for feat in feature_names}
        node = leaf
        while node != 0:
            if node in left:
                parent = int(np.where(left == node)[0][0])
                side = '<='
            else:
                parent = int(np.where(right == node)[0][0])
                side = '>'
            feature = feats[parent]
            threshold = thr[parent]
            if feature in path_constraints:
                path_constraints[feature].append((side, threshold))
            node = parent

        # Predicted class
        class_counts = list(value[leaf][0])
        pred_class = int(class_counts.index(max(class_counts)))

        # Compatible indices for each feature
        feat_indices = {}
        valid = True
        for feat in feature_names:
            idxs = get_compatible_indices(
                path_constraints[feat],
                intervals_per_feature[feat]
            )
            if not idxs:
                valid = False
                break
            feat_indices[feat] = idxs

        if not valid:
            continue

        s1_min, s1_max = min(feat_indices['ip_proto']), max(feat_indices['ip_proto'])
        s2_min, s2_max = min(feat_indices['src_port']), max(feat_indices['src_port'])
        s3_min, s3_max = min(feat_indices['dst_port']), max(feat_indices['dst_port'])

        entry_key = (s1_min, s1_max, s2_min, s2_max, s3_min, s3_max)
        if entry_key in seen:
            continue
        seen.add(entry_key)

        action = class_to_action[pred_class]
        nhop, port = action['nhop'], action['port']

        cmd = (f'{CLI} <<< "table_add MyIngress.ipv4_exact '
               f'MyIngress.ipv4_forward '
               f'{s1_min}->{s1_max} {s2_min}->{s2_max} {s3_min}->{s3_max} '
               f'=> {nhop} {port} 1"   '
               f'# Class {pred_class} ({class_names[pred_class]})')
        commands.append(cmd)

    return commands


# Generate commands for the L3 tree (real UNSW)
commands_L3 = generate_all_table_add_commands(
    clf_L3, FEATURE_NAMES, intervals_L3, CLASS_TO_ACTION
)

print(f"Total generated commands: {len([c for c in commands_L3 if c.startswith('simple_switch')])}")
print(f"(of which are feature tables: {sum(1 for c in commands_L3 if 'feature' in c and 'select' in c)})")
print(f"(and ipv4_exact decision table:  {sum(1 for c in commands_L3 if 'ipv4_exact' in c)})")

Total generated commands: 18
(of which are feature tables: 10)
(and ipv4_exact decision table:  9)


In [14]:
# =============================================================================
# SECTION 9.2: DISPLAY AND SAVE COMMANDS
# =============================================================================

print("=" * 70)
print("GENERATED table_add COMMANDS (L3 tree (real UNSW)):")
print("=" * 70)
for cmd in commands_L3:
    print(cmd)

# Save shell script
script_header = """#!/bin/bash
# Script automatically generated by ML_Pipeline_DT.ipynb
# Decision Tree L3 (max_depth=3)
# Dataset: synthetic representative of Sivanathan et al. (2018)
# (requires BMv2 switch to be running on thrift-port 9090)

"""
with open('rules-extracted.txt', 'w', encoding='utf-8') as f:
    f.write(script_header)
    for cmd in commands_L3:
        f.write(cmd + '\n')

print(f"\nScript saved: rules-extracted.txt")

GENERATED table_add COMMANDS (L3 tree (real UNSW)):
# ─── Table 1: ip_proto (Feature 1) ───
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature1_exact MyIngress.set_actionselect1 0->255 => 1 1"

# ─── Table 2: src_port (Feature 2) ───
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature2_exact MyIngress.set_actionselect2 0->547 => 1 1"
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature2_exact MyIngress.set_actionselect2 548->1899 => 2 1"
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature2_exact MyIngress.set_actionselect2 1900->3071 => 3 1"
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature2_exact MyIngress.set_actionselect2 3072->49280 => 4 1"
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature2_exact MyIngress.set_actionselect2 49281->60633 => 5 1"
simple_switch_CLI --thrift-port 9090 <<< "table_add MyIngress.feature2_exact MyIngress.set_actionselect2 60634->65535 => 6 1"

# 